In [1]:
# ========== 导入 + 本地 Ollama（OpenAI 兼容端点）==========

# 从 openai 导入 OpenAI 客户端：用同一套 chat.completions API 调本地模型
from openai import OpenAI
# 导入 gradio：快速做浏览器聊天界面
import gradio as gr


# base_url 指向本机 Ollama 的 OpenAI 兼容 /v1；api_key 本地可填任意非空占位（字符串保持原样）
ollama = OpenAI(
    base_url="http://localhost:11434/v1",
    api_key="ollama"
)


In [2]:
# ========== System Prompt + 流式 chat 生成器 ==========

# system_message：人设（幽默、Hinglish、罗马字母）。英文/Hinglish 正文是发给模型的指令，不翻译
system_message = "you are an assisstant with a great sense of humour. Your replies often contains jokes. you can talk in hindi and english. You always reply in roman script. You reply in Hinglish when user replies in hindi"

def chat(message, history):
    # Gradio messages → 只保留 role/content，避免多余字段进 API
    history = [{"role":h["role"], "content": h["content"]} for h in history]

    # 标准三拼：system + 历史 + 当前 user
    messages = [{"role":"system", "content":system_message}] + history + [{"role":"user", "content":message}]

    # stream=True：服务端边生成边推；返回可迭代的 chunk 流
    stream = ollama.chat.completions.create(
        model="gemma4:31b-cloud",
        messages=messages,
        stream=True
    )

    # 累积已生成文本；每来一个 delta 就 yield 全文，Gradio 才能「打字机」刷新
    response = ""
    for chunk in stream:
        # delta.content 可能是 None（某些 chunk 只有 role/结束标记），用 or '' 兜底
        response += chunk.choices[0].delta.content or ''
        yield response


In [3]:
# ========== 启动 Gradio 聊天界面 ==========

# ChatInterface 把输入/历史交给 fn=chat；type="messages" 使用消息列表格式
# 前提：本机 Ollama 在跑，且已具备 model 字符串对应的模型
gr.ChatInterface(fn=chat, type="messages").launch()


* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


In [ ]:
# （空单元格保留：原先就是空的，不改逻辑、不删格）
